In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import os
import dagshub

c:\Users\avanindra Bose\Twitter Emotion Detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI"))
dagshub.init(repo_owner='AvanindraBose', repo_name='Twitter-Sentiment-Detection', mlflow=True)


Accessing as AvanindraBose

Initialized MLflow to track repo "AvanindraBose/Twitter-Sentiment-Detection"

Repository AvanindraBose/Twitter-Sentiment-Detection initialized!

In [3]:
# Load the data
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [4]:
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(removing_punctuations)
        df['content'] = df['content'].apply(removing_urls)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

# Normalize the text data
df = normalize_text(df)

In [5]:
x = df['sentiment'].isin(['happiness','sadness'])
df = df[x]

In [6]:
df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})

C:\Users\avanindra Bose\AppData\Local\Temp\ipykernel_39332\1508851655.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})


In [7]:
# Define feature extraction methods
vectorizers = {
    'BoW': CountVectorizer(),
    'TF-IDF': TfidfVectorizer()
}

# Define algorithms
algorithms = {
    'LogisticRegression': LogisticRegression(),
    'MultinomialNB': MultinomialNB(),
    'XGBoost': XGBClassifier(),
    'RandomForest': RandomForestClassifier(),
    'GradientBoosting': GradientBoostingClassifier()
}

In [8]:
vectorized_data = {}

for vec_name , vectorizer in vectorizers.items():
    X = vectorizer.fit_transform(df['content'])
    y = df['sentiment']
    
    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    vectorized_data[vec_name] = (X_train, X_test, y_train, y_test)

In [10]:
mlflow.set_experiment("Comparing BoW and TF-IDF with Multiple Algorithms")

with mlflow.start_run(run_name = "All Experiments") as parent_run:
    data_df = mlflow.data.from_pandas(df)
    mlflow.log_input(data_df , "Dataset")
    for algo_name,alg in algorithms.items():
        for vec_name , vec_data in vectorized_data.items():
            with mlflow.start_run(run_name = f"{algo_name} with {vec_name}", nested=True) as child_run:
                X_train,X_Test,y_train,y_test = vectorized_data[vec_name]
                mlflow.log_param("vectorizer", vec_name)
                mlflow.log_param("algorithm", algo_name)
                mlflow.log_param("test_size", 0.2)

                model = alg
                model.fit(X_train,y_train)

                if algo_name == 'LogisticRegression':
                    mlflow.log_param("C", model.C)
                elif algo_name == 'MultinomialNB':
                    mlflow.log_param("alpha", model.alpha)
                elif algo_name == 'XGBoost':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("learning_rate", model.learning_rate)
                elif algo_name == 'RandomForest':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("max_depth", model.max_depth)
                elif algo_name == 'GradientBoosting':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("learning_rate", model.learning_rate)
                    mlflow.log_param("max_depth", model.max_depth)
                
                y_pred = model.predict(X_test)
                accuracy = accuracy_score(y_test, y_pred)
                precision = precision_score(y_test, y_pred)
                recall = recall_score(y_test, y_pred)
                f1 = f1_score(y_test, y_pred)
                
                # Log evaluation metrics
                mlflow.log_metric("accuracy", accuracy)
                mlflow.log_metric("precision", precision)
                mlflow.log_metric("recall", recall)
                mlflow.log_metric("f1_score", f1)

                mlflow.sklearn.log_model(model, "model")
                
                # Save and log the notebook
                notebook_path = "exp2_bow_vs_tfidf.ipynb"
                os.system(f"jupyter nbconvert --to notebook --execute --inplace {notebook_path}")
                mlflow.log_artifact(notebook_path)
                
                # Print the results for verification
                print(f"Algorithm: {algo_name}, Feature Engineering: {vec_name}")
                print(f"Accuracy: {accuracy}")
                print(f"Precision: {precision}")
                print(f"Recall: {recall}")
                print(f"F1 Score: {f1}")



2026/04/12 20:12:18 INFO mlflow.tracking.fluent: Experiment with name 'Comparing BoW and TF-IDF with Multiple Algorithms' does not exist. Creating a new experiment.
c:\Users\avanindra Bose\Twitter Emotion Detection\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/12 20:12:28 WARNING mlflow.models.model: 

Algorithm: LogisticRegression, Feature Engineering: BoW
Accuracy: 0.7816867469879518
Precision: 0.8002136752136753
Recall: 0.7379310344827587
F1 Score: 0.7678113787801127
🏃 View run LogisticRegression with BoW at: https://dagshub.com/AvanindraBose/Twitter-Sentiment-Detection.mlflow/#/experiments/1/runs/a655a2d888194213b59e5ada0f2a71a8
🧪 View experiment at: https://dagshub.com/AvanindraBose/Twitter-Sentiment-Detection.mlflow/#/experiments/1


2026/04/12 20:13:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/12 20:13:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/12 20:13:18 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Algorithm: LogisticRegression, Feature Engineering: TF-IDF
Accuracy: 0.7942168674698795
Precision: 0.777882797731569
Recall: 0.8108374384236453
F1 Score: 0.79401833092137
🏃 View run LogisticRegression with TF-IDF at: https://dagshub.com/AvanindraBose/Twitter-Sentiment-Detection.mlflow/#/experiments/1/runs/26fdf5a76a3f49b19d5ce01133986b71
🧪 View experiment at: https://dagshub.com/AvanindraBose/Twitter-Sentiment-Detection.mlflow/#/experiments/1


2026/04/12 20:13:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/12 20:13:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/12 20:13:47 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


...

Algorithm: RandomForest, Feature Engineering: TF-IDF
Accuracy: 0.7759036144578313
Precision: 0.7728174603174603
Recall: 0.767487684729064
F1 Score: 0.7701433514582303
🏃 View run RandomForest with TF-IDF at: https://dagshub.com/AvanindraBose/Twitter-Sentiment-Detection.mlflow/#/experiments/1/runs/73384ae2ec1e4cf6b41dc29b783c81e5
🧪 View experiment at: https://dagshub.com/AvanindraBose/Twitter-Sentiment-Detection.mlflow/#/experiments/1


2026/04/12 20:21:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/12 20:21:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/12 20:21:53 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Algorithm: GradientBoosting, Feature Engineering: BoW
Accuracy: 0.5219277108433735
Precision: 0.9259259259259259
Recall: 0.024630541871921183
F1 Score: 0.04798464491362764
🏃 View run GradientBoosting with BoW at: https://dagshub.com/AvanindraBose/Twitter-Sentiment-Detection.mlflow/#/experiments/1/runs/c5c8086927b144259edfdb006df227f8
🧪 View experiment at: https://dagshub.com/AvanindraBose/Twitter-Sentiment-Detection.mlflow/#/experiments/1


2026/04/12 20:23:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/12 20:23:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/12 20:23:14 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


Algorithm: GradientBoosting, Feature Engineering: TF-IDF
Accuracy: 0.723855421686747
Precision: 0.8027397260273973
Recall: 0.5773399014778325
F1 Score: 0.6716332378223495
🏃 View run GradientBoosting with TF-IDF at: https://dagshub.com/AvanindraBose/Twitter-Sentiment-Detection.mlflow/#/experiments/1/runs/f6d5a1f0cdaa432ebf4c9638dd3e4d5d
🧪 View experiment at: https://dagshub.com/AvanindraBose/Twitter-Sentiment-Detection.mlflow/#/experiments/1
🏃 View run All Experiments at: https://dagshub.com/AvanindraBose/Twitter-Sentiment-Detection.mlflow/#/experiments/1/runs/d9cd19cc036f4c18990e8b0c11ef6230
🧪 View experiment at: https://dagshub.com/AvanindraBose/Twitter-Sentiment-Detection.mlflow/#/experiments/1
